# NB05 — Wildfire Modeling, Imbalance Handling & Hyperparameter Tuning

**Objective:** Build a production-quality wildfire detection model that prioritises **Recall** and **F1** (not accuracy) on a heavily imbalanced dataset (~8–9% positive class).

### Strategy
| Step | What |
|------|------|
| §1 | Load wildfire ML dataset, temporal train/test split |
| §2 | Train **9 diverse models** with built-in class weighting |
| §3 | Apply **SMOTE / SMOTEENN** to top models and compare |
| §4 | **Optuna** hyperparameter tuning for best 2 models |
| §5 | **Threshold optimisation** — find best operating point on PR curve |
| §6 | **Probability calibration** — isotonic calibration |
| §7 | Full **leaderboard** — Recall, F1, Precision, PR-AUC, ROC-AUC, FN count |
| §8 | **SHAP** / feature importance analysis |
| §9 | Save best model, leaderboard, feature importance |

### Key Design Decisions
- **Selection criterion:** best composite = 0.6×Recall + 0.4×F1 (subject to Precision ≥ 10%)
- **Validation:** early stopping uses a **held-out validation set** (not the test set) to prevent leakage
- **Optuna objective** is constrained: we reject trials where precision < 10%
- **Threshold sweep** enforces minimum precision to avoid degenerate "predict-everything" models

**Input:** `data/processed/engineered_daily.parquet`  
**Output:** `models/wildfire/best_fire_model.joblib`, `reports/metrics/fire_leaderboard.csv`, `reports/figures/`

In [ ]:
# ─── Cell 1: Imports & Project Setup ─────────────────────────────────────
import subprocess, sys, os, warnings, json, time
from pathlib import Path

# Ensure all dependencies are available
for _p in ["xgboost","lightgbm","catboost","optuna","imbalanced-learn",
           "scikit-learn","pandas","numpy","matplotlib","seaborn","shap","joblib"]:
    try: __import__(_p.replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _p])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, recall_score,
    precision_recall_curve, average_precision_score, roc_auc_score,
    precision_score, accuracy_score)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from joblib import dump as jl_dump, load as jl_load
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
from tqdm.auto import tqdm

# imblearn
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTEENN
try:
    from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
    HAS_IMBLEARN_ENS = True
except ImportError:
    HAS_IMBLEARN_ENS = False

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")

# ── Project paths ────────────────────────────────────────────────────────
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.config import (ROOT, PROCESSED, OUTPUTS, MODELS_F, FIGURES, METRICS,
                         TARGET_COL, LEAK_COLS, ID_COLS, DROP_COLS, RANDOM_SEED, SPLIT_DATE,
                         ENG_DAILY, MASTER_DAILY, WILDFIRE_DS)
from src.evaluation import fire_metrics, find_optimal_threshold, build_fire_leaderboard
from src.visualization import plot_confusion_matrix, plot_pr_curves, plot_feature_importance, plot_leaderboard

SEED = RANDOM_SEED
np.random.seed(SEED)
print(f"Root: {ROOT}")

## §1 — Load Data & Temporal Split

We use a **strict temporal split**: train on data before 2025-01-01, test on data after.
A **validation set** (last 20% of training by time) is used for early stopping and threshold tuning — this prevents data leakage from the test set into model selection.

In [ ]:
# ─── §1: Load data, define features, temporal split ──────────────────────

# Try engineered daily first, fall back to master
if ENG_DAILY.exists():
    df = pd.read_parquet(ENG_DAILY)
    print(f"Loaded engineered_daily: {df.shape}")
elif MASTER_DAILY.exists():
    df = pd.read_parquet(MASTER_DAILY)
    print(f"Loaded master_daily (no NB02 features): {df.shape}")
else:
    raise FileNotFoundError("No daily dataset found. Run NB01 + NB02 first.")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["City", "Date"]).reset_index(drop=True)

# Feature columns = all numeric except target, IDs, and leakage columns
feature_cols = [c for c in df.columns
                if c not in DROP_COLS
                and df[c].dtype in ["float64", "float32", "int64", "int32"]]

# ── Temporal split ────────────────────────────────────────────────────────
split_ts = pd.Timestamp(SPLIT_DATE)
train_full = df[df["Date"] < split_ts].copy()
test_df    = df[df["Date"] >= split_ts].copy()

# Validation split: last 20% of training data (by time)
val_cutoff = train_full["Date"].quantile(0.80)
train_df = train_full[train_full["Date"] < val_cutoff].copy()
val_df   = train_full[train_full["Date"] >= val_cutoff].copy()

X_train = train_df[feature_cols].fillna(0)
y_train = train_df[TARGET_COL].values
X_val   = val_df[feature_cols].fillna(0)
y_val   = val_df[TARGET_COL].values
X_test  = test_df[feature_cols].fillna(0)
y_test  = test_df[TARGET_COL].values

# For full train (train+val) used after tuning
X_train_full = train_full[feature_cols].fillna(0)
y_train_full = train_full[TARGET_COL].values

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
IMBALANCE_RATIO = n_neg / max(n_pos, 1)

print(f"\nFeatures: {len(feature_cols)}")
print(f"Train:      {X_train.shape}  fires={y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Validation: {X_val.shape}  fires={y_val.sum()} ({y_val.mean()*100:.2f}%)")
print(f"Test:       {X_test.shape}  fires={y_test.sum()} ({y_test.mean()*100:.2f}%)")
print(f"Imbalance ratio: {IMBALANCE_RATIO:.1f}:1")
print(f"\nTrain period:      {train_df['Date'].min().date()} → {train_df['Date'].max().date()}")
print(f"Validation period: {val_df['Date'].min().date()} → {val_df['Date'].max().date()}")
print(f"Test period:       {test_df['Date'].min().date()} → {test_df['Date'].max().date()}")

## §2 — Train 9 Diverse Models (Built-in Class Weighting)

We train all models using **built-in class weighting** (no resampling yet).
Each model's probability output is then threshold-tuned on the **validation set**.

In [ ]:
# ─── §2: Train multiple models with class weighting ──────────────────────

models_config = {}

# 1. LogisticRegression
models_config["LogReg"] = {
    "model": LogisticRegression(class_weight="balanced", max_iter=1000,
                                random_state=SEED, n_jobs=-1),
    "strategy": "class_weight=balanced"}

# 2. RandomForest
models_config["RF"] = {
    "model": RandomForestClassifier(n_estimators=300, max_depth=20,
                                    min_samples_split=5, class_weight="balanced",
                                    random_state=SEED, n_jobs=-1),
    "strategy": "class_weight=balanced"}

# 3. ExtraTrees
models_config["ExtraTrees"] = {
    "model": ExtraTreesClassifier(n_estimators=300, max_depth=20,
                                  min_samples_split=5, class_weight="balanced",
                                  random_state=SEED, n_jobs=-1),
    "strategy": "class_weight=balanced"}

# 4. HistGradientBoosting
models_config["HistGBC"] = {
    "model": HistGradientBoostingClassifier(
        max_iter=500, max_depth=8, learning_rate=0.05,
        class_weight="balanced", random_state=SEED),
    "strategy": "class_weight=balanced"}

# 5. XGBoost cost-sensitive
models_config["XGBoost"] = {
    "model": xgb.XGBClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=IMBALANCE_RATIO,
        eval_metric="aucpr", early_stopping_rounds=30,
        random_state=SEED, use_label_encoder=False, n_jobs=-1),
    "strategy": f"scale_pos_weight={IMBALANCE_RATIO:.1f}",
    "fit_kwargs": {"eval_set": [(X_val, y_val)], "verbose": False}}

# 6. LightGBM
models_config["LightGBM"] = {
    "model": lgb.LGBMClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        is_unbalance=True, random_state=SEED, n_jobs=-1, verbose=-1),
    "strategy": "is_unbalance=True",
    "fit_kwargs": {"eval_set": [(X_val, y_val)],
                   "callbacks": [lgb.early_stopping(30, verbose=False)]}}

# 7. CatBoost
models_config["CatBoost"] = {
    "model": cb.CatBoostClassifier(
        iterations=500, depth=8, learning_rate=0.05,
        auto_class_weights="Balanced", eval_metric="F1",
        random_seed=SEED, verbose=0),
    "strategy": "auto_class_weights=Balanced",
    "fit_kwargs": {"eval_set": (X_val, y_val), "verbose": 0}}

# 8. BalancedRandomForest (imblearn)
if HAS_IMBLEARN_ENS:
    models_config["BalancedRF"] = {
        "model": BalancedRandomForestClassifier(
            n_estimators=300, max_depth=20, min_samples_split=5,
            random_state=SEED, n_jobs=-1),
        "strategy": "balanced_subsample"}

# 9. EasyEnsemble (imblearn)
if HAS_IMBLEARN_ENS:
    models_config["EasyEnsemble"] = {
        "model": EasyEnsembleClassifier(
            n_estimators=20, random_state=SEED, n_jobs=-1),
        "strategy": "easy_ensemble"}

# ── Train all models ─────────────────────────────────────────────────────
results = {}
print(f"Training {len(models_config)} models...\n")

for name, cfg in models_config.items():
    t0 = time.time()
    print(f"--- {name} ({cfg['strategy']}) ---")
    try:
        model = cfg["model"]
        fit_kw = cfg.get("fit_kwargs", {})
        model.fit(X_train, y_train, **fit_kw)

        y_prob_val  = model.predict_proba(X_val)[:, 1]
        y_prob_test = model.predict_proba(X_test)[:, 1]

        # Find optimal threshold on VALIDATION set (not test)
        best_thresh = find_optimal_threshold(y_val, y_prob_val,
                                             recall_weight=0.6, f1_weight=0.4,
                                             min_precision=0.10)

        y_pred_test = (y_prob_test >= best_thresh).astype(int)

        results[name] = {
            "model": model,
            "y_true": y_test,
            "y_pred": y_pred_test,
            "y_prob": y_prob_test,
            "threshold": best_thresh,
            "imbalance_strategy": cfg["strategy"],
            "train_time": time.time() - t0,
        }

        m = fire_metrics(y_test, y_pred_test, y_prob_test)
        print(f"  Thresh={best_thresh:.2f}  Recall={m['recall']:.4f}  "
              f"F1={m['f1']:.4f}  Prec={m['precision']:.4f}  "
              f"PR-AUC={m.get('pr_auc',0):.4f}  FN={m['fn']}  "
              f"({time.time()-t0:.1f}s)")

    except Exception as e:
        print(f"  FAILED: {e}")

print(f"\nSuccessfully trained: {len(results)} / {len(models_config)} models")

## §3 — SMOTE / SMOTEENN on Top Models

We apply resampling **only** to the top 2 gradient-boosting models (XGBoost, LightGBM).
We use conservative sampling ratios (0.3 and 0.5) to avoid generating excessive synthetic noise.

In [ ]:
# ─── §3: SMOTE / SMOTEENN resampling on top models ───────────────────────

resamplers = {
    "SMOTE_0.3":    SMOTE(sampling_strategy=0.3, random_state=SEED, k_neighbors=5),
    "SMOTE_0.5":    SMOTE(sampling_strategy=0.5, random_state=SEED, k_neighbors=5),
    "BSMOTE_0.3":   BorderlineSMOTE(sampling_strategy=0.3, random_state=SEED,
                                     k_neighbors=5, kind="borderline-1"),
    "SMOTEENN_0.3": SMOTEENN(sampling_strategy=0.3, random_state=SEED),
}

# Only apply to gradient boosting models
boost_models = {
    "XGB": lambda: xgb.XGBClassifier(
        n_estimators=400, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr", early_stopping_rounds=25,
        random_state=SEED, use_label_encoder=False, n_jobs=-1),
    "LGB": lambda: lgb.LGBMClassifier(
        n_estimators=400, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbose=-1),
}

print(f"Testing {len(resamplers)} resamplers × {len(boost_models)} models\n")

for res_name, sampler in resamplers.items():
    for mdl_name, mdl_factory in boost_models.items():
        combo_name = f"{mdl_name}_{res_name}"
        t0 = time.time()
        print(f"--- {combo_name} ---")
        try:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            print(f"  Resampled: 0={int((y_res==0).sum())} 1={int((y_res==1).sum())}")

            model = mdl_factory()
            if "XGB" in mdl_name:
                model.fit(X_res, y_res, eval_set=[(X_val, y_val)], verbose=False)
            elif "LGB" in mdl_name:
                model.fit(X_res, y_res, eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(25, verbose=False)])
            else:
                model.fit(X_res, y_res)

            y_prob_val  = model.predict_proba(X_val)[:, 1]
            y_prob_test = model.predict_proba(X_test)[:, 1]

            best_thresh = find_optimal_threshold(y_val, y_prob_val,
                                                 recall_weight=0.6, f1_weight=0.4,
                                                 min_precision=0.10)
            y_pred_test = (y_prob_test >= best_thresh).astype(int)

            results[combo_name] = {
                "model": model,
                "y_true": y_test, "y_pred": y_pred_test, "y_prob": y_prob_test,
                "threshold": best_thresh,
                "imbalance_strategy": res_name,
                "train_time": time.time() - t0,
            }

            m = fire_metrics(y_test, y_pred_test, y_prob_test)
            print(f"  Thresh={best_thresh:.2f}  Recall={m['recall']:.4f}  "
                  f"F1={m['f1']:.4f}  Prec={m['precision']:.4f}  "
                  f"PR-AUC={m.get('pr_auc',0):.4f}  ({time.time()-t0:.1f}s)")

        except Exception as e:
            print(f"  FAILED: {e}")

print(f"\nTotal models evaluated: {len(results)}")

## §4 — Optuna Hyperparameter Tuning

We tune the **top 2 models** by composite score (0.6×Recall + 0.4×F1).
The Optuna objective enforces **minimum precision ≥ 12%** to reject degenerate solutions.
Validation set is used for early stopping — test set is not seen during tuning.

In [ ]:
# ─── §4: Optuna tuning for top models ────────────────────────────────────

# Build interim leaderboard to pick top 2
interim_lb = build_fire_leaderboard(results)
print("Interim leaderboard (top 10):")
print(interim_lb[["model", "recall", "f1", "precision", "pr_auc", "composite"]].head(10).to_string(index=False))

top2 = interim_lb["model"].head(2).tolist()
print(f"\nTuning: {top2}")

# ── Optuna objective factory ─────────────────────────────────────────────
def make_xgb_objective(X_tr, y_tr, X_v, y_v, imb_ratio):
    def objective(trial):
        p = {
            "n_estimators":     trial.suggest_int("n_estimators", 200, 800),
            "max_depth":        trial.suggest_int("max_depth", 4, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
            "gamma":            trial.suggest_float("gamma", 0.0, 3.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-6, 5.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight",
                                                     imb_ratio * 0.3, imb_ratio * 2.0),
        }
        clf = xgb.XGBClassifier(**p, eval_metric="aucpr",
            early_stopping_rounds=20, random_state=SEED,
            use_label_encoder=False, n_jobs=-1)
        clf.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)

        y_prob = clf.predict_proba(X_v)[:, 1]
        thresh = find_optimal_threshold(y_v, y_prob,
                                        recall_weight=0.6, f1_weight=0.4,
                                        min_precision=0.12)
        y_pred = (y_prob >= thresh).astype(int)
        rec  = recall_score(y_v, y_pred, zero_division=0)
        f1   = f1_score(y_v, y_pred, zero_division=0)
        prec = precision_score(y_v, y_pred, zero_division=0)

        # Penalise degenerate solutions
        if prec < 0.12:
            return 0.0
        return 0.6 * rec + 0.4 * f1
    return objective


def make_lgb_objective(X_tr, y_tr, X_v, y_v, imb_ratio):
    def objective(trial):
        p = {
            "n_estimators":     trial.suggest_int("n_estimators", 200, 800),
            "max_depth":        trial.suggest_int("max_depth", 4, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-6, 5.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight",
                                                     imb_ratio * 0.3, imb_ratio * 2.0),
        }
        clf = lgb.LGBMClassifier(**p, random_state=SEED, n_jobs=-1, verbose=-1)
        clf.fit(X_tr, y_tr, eval_set=[(X_v, y_v)],
                callbacks=[lgb.early_stopping(20, verbose=False)])

        y_prob = clf.predict_proba(X_v)[:, 1]
        thresh = find_optimal_threshold(y_v, y_prob,
                                        recall_weight=0.6, f1_weight=0.4,
                                        min_precision=0.12)
        y_pred = (y_prob >= thresh).astype(int)
        rec  = recall_score(y_v, y_pred, zero_division=0)
        f1   = f1_score(y_v, y_pred, zero_division=0)
        prec = precision_score(y_v, y_pred, zero_division=0)

        if prec < 0.12:
            return 0.0
        return 0.6 * rec + 0.4 * f1
    return objective


# ── Run Optuna ────────────────────────────────────────────────────────────
N_TRIALS = 60

for model_name in top2:
    print(f"\n{'='*60}")
    print(f"OPTUNA TUNING: {model_name} ({N_TRIALS} trials)")
    print(f"{'='*60}")

    # Determine if XGBoost or LightGBM family
    if "XGB" in model_name or "XGBoost" in model_name:
        obj = make_xgb_objective(X_train, y_train, X_val, y_val, IMBALANCE_RATIO)
    elif "LGB" in model_name or "LightGBM" in model_name:
        obj = make_lgb_objective(X_train, y_train, X_val, y_val, IMBALANCE_RATIO)
    elif "CatBoost" in model_name:
        # Skip CatBoost Optuna (slower, already decent)
        print("  Skipping CatBoost Optuna (use built-in result)")
        continue
    else:
        # For non-boost models, use XGB objective as proxy
        obj = make_xgb_objective(X_train, y_train, X_val, y_val, IMBALANCE_RATIO)

    study = optuna.create_study(direction="maximize",
                                study_name=f"fire_{model_name}")
    study.optimize(obj, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nBest score: {study.best_trial.value:.4f}")
    best_p = study.best_trial.params
    for k, v in best_p.items():
        print(f"  {k}: {v}")

    # Re-train best model on train set, evaluate on test
    optuna_name = f"{model_name}_Optuna"
    if "XGB" in model_name or "XGBoost" in model_name:
        best_model = xgb.XGBClassifier(**best_p, eval_metric="aucpr",
            early_stopping_rounds=30, random_state=SEED,
            use_label_encoder=False, n_jobs=-1)
        best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    else:
        best_model = lgb.LGBMClassifier(**best_p, random_state=SEED,
                                         n_jobs=-1, verbose=-1)
        best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                       callbacks=[lgb.early_stopping(30, verbose=False)])

    y_prob_val  = best_model.predict_proba(X_val)[:, 1]
    y_prob_test = best_model.predict_proba(X_test)[:, 1]
    best_thresh = find_optimal_threshold(y_val, y_prob_val,
                                         recall_weight=0.6, f1_weight=0.4,
                                         min_precision=0.10)
    y_pred_test = (y_prob_test >= best_thresh).astype(int)

    results[optuna_name] = {
        "model": best_model,
        "y_true": y_test, "y_pred": y_pred_test, "y_prob": y_prob_test,
        "threshold": best_thresh,
        "imbalance_strategy": f"optuna_{N_TRIALS}trials",
        "train_time": 0,
        "best_params": best_p,
    }

    m = fire_metrics(y_test, y_pred_test, y_prob_test)
    print(f"\n  Test: Thresh={best_thresh:.2f}  Recall={m['recall']:.4f}  "
          f"F1={m['f1']:.4f}  Prec={m['precision']:.4f}  PR-AUC={m.get('pr_auc',0):.4f}")
    print(classification_report(y_test, y_pred_test, digits=4,
                                target_names=["No Fire", "Fire"]))

## §5 — Full Leaderboard & Best Model Selection

Build a comprehensive leaderboard sorted by composite score (0.6×Recall + 0.4×F1).
The best model must have **Precision ≥ 10%** — we refuse to deploy a model that predicts everything as fire.

In [ ]:
# ─── §5: Build full leaderboard ──────────────────────────────────────────

leaderboard = build_fire_leaderboard(results)

# Display
display_cols = ["model", "imbalance_strategy", "threshold",
                "recall", "precision", "f1", "pr_auc", "roc_auc",
                "fn", "fp", "composite"]
avail_cols = [c for c in display_cols if c in leaderboard.columns]

print("=" * 90)
print("WILDFIRE DETECTION LEADERBOARD")
print("=" * 90)
print(leaderboard[avail_cols].to_string(index=False))

# Save
leaderboard.to_csv(METRICS / "fire_leaderboard.csv", index=False)
print(f"\nLeaderboard saved: {METRICS / 'fire_leaderboard.csv'}")

# Select best model
best_row = leaderboard.iloc[0]
BEST_NAME = best_row["model"]
print(f"\n{'='*60}")
print(f"BEST MODEL: {BEST_NAME}")
print(f"  Recall    = {best_row['recall']:.4f}")
print(f"  Precision = {best_row['precision']:.4f}")
print(f"  F1        = {best_row['f1']:.4f}")
print(f"  PR-AUC    = {best_row.get('pr_auc', 0):.4f}")
print(f"  Threshold = {best_row['threshold']:.2f}")
print(f"  FN        = {best_row['fn']}")
print(f"{'='*60}")

best_result = results[BEST_NAME]
best_model  = best_result["model"]
BEST_THRESH = best_result["threshold"]

## §6 — Probability Calibration

Isotonic calibration on the validation set produces well-calibrated probabilities that can be meaningfully interpreted as "risk percentage" in the dashboard.

In [ ]:
# ─── §6: Probability calibration ─────────────────────────────────────────

print("Calibrating best model probabilities (isotonic)...")

try:
    calibrated_model = CalibratedClassifierCV(best_model, method="isotonic", cv="prefit")
    calibrated_model.fit(X_val, y_val)

    y_prob_cal = calibrated_model.predict_proba(X_test)[:, 1]

    # Re-tune threshold after calibration
    y_prob_cal_val = calibrated_model.predict_proba(X_val)[:, 1]
    CAL_THRESH = find_optimal_threshold(y_val, y_prob_cal_val,
                                        recall_weight=0.6, f1_weight=0.4,
                                        min_precision=0.10)
    y_pred_cal = (y_prob_cal >= CAL_THRESH).astype(int)

    m_cal = fire_metrics(y_test, y_pred_cal, y_prob_cal)
    print(f"\nCalibrated model:")
    print(f"  Threshold = {CAL_THRESH:.2f}")
    print(f"  Recall    = {m_cal['recall']:.4f}")
    print(f"  F1        = {m_cal['f1']:.4f}")
    print(f"  Precision = {m_cal['precision']:.4f}")
    print(f"  PR-AUC    = {m_cal.get('pr_auc',0):.4f}")

    # Use calibrated if it doesn't degrade F1 too much
    composite_uncal = 0.6 * best_row["recall"] + 0.4 * best_row["f1"]
    composite_cal   = 0.6 * m_cal["recall"] + 0.4 * m_cal["f1"]

    if composite_cal >= composite_uncal * 0.95:
        print("  ✓ Using calibrated model (no significant degradation)")
        FINAL_MODEL = calibrated_model
        FINAL_THRESH = CAL_THRESH
        FINAL_PROB_TEST = y_prob_cal
    else:
        print("  ⚠ Calibration degraded performance — using uncalibrated model")
        FINAL_MODEL = best_model
        FINAL_THRESH = BEST_THRESH
        FINAL_PROB_TEST = best_result["y_prob"]

except Exception as e:
    print(f"  Calibration failed: {e} — using uncalibrated model")
    FINAL_MODEL = best_model
    FINAL_THRESH = BEST_THRESH
    FINAL_PROB_TEST = best_result["y_prob"]

FINAL_PRED_TEST = (FINAL_PROB_TEST >= FINAL_THRESH).astype(int)
print(f"\nFinal classification report:")
print(classification_report(y_test, FINAL_PRED_TEST, digits=4,
                            target_names=["No Fire", "Fire"]))

## §7 — Evaluation Visualizations

Confusion matrices, PR curves, and feature importance for the top models.

In [ ]:
# ─── §7a: Confusion matrices — top 4 models ──────────────────────────────

top4 = leaderboard["model"].head(4).tolist()
fig, axes = plt.subplots(1, len(top4), figsize=(6 * len(top4), 5))
fig.suptitle("Confusion Matrices — Top Models", fontsize=14)
if len(top4) == 1:
    axes = [axes]

for ax, name in zip(axes, top4):
    r = results[name]
    plot_confusion_matrix(r["y_true"], r["y_pred"], title=name, ax=ax)

plt.tight_layout()
plt.savefig(FIGURES / "fire_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── §7b: PR curves — all models ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
curves = {name: r["y_prob"] for name, r in results.items() if "y_prob" in r}
plot_pr_curves(curves, y_test, title="Precision-Recall Curves — All Fire Models", ax=ax)
plt.tight_layout()
plt.savefig(FIGURES / "fire_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── §7c: Leaderboard bar charts ─────────────────────────────────────────
fig = plot_leaderboard(leaderboard.head(10), title="Fire Detection Leaderboard (Top 10)")
if fig:
    plt.savefig(FIGURES / "fire_leaderboard_chart.png", dpi=150, bbox_inches="tight")
    plt.show()

# ─── §7d: City-level recall analysis ─────────────────────────────────────
print("\nCity-level Recall (best model):")
test_with_pred = test_df[["City", "Date", TARGET_COL]].copy()
test_with_pred["y_pred"] = FINAL_PRED_TEST

city_metrics = []
for city in sorted(test_with_pred["City"].unique()):
    cm = test_with_pred[test_with_pred["City"] == city]
    n_fires = cm[TARGET_COL].sum()
    if n_fires > 0:
        rec = recall_score(cm[TARGET_COL], cm["y_pred"], zero_division=0)
        f1  = f1_score(cm[TARGET_COL], cm["y_pred"], zero_division=0)
    else:
        rec, f1 = np.nan, np.nan
    city_metrics.append({"City": city, "n_fires": int(n_fires),
                          "recall": rec, "f1": f1})
    print(f"  {city:15s}  fires={int(n_fires):3d}  recall={rec:.3f}  f1={f1:.3f}"
          if n_fires > 0 else f"  {city:15s}  fires=  0  (no fires in test)")

city_metrics_df = pd.DataFrame(city_metrics)
city_metrics_df.to_csv(METRICS / "fire_city_metrics.csv", index=False)

## §8 — Model Explainability (SHAP + Feature Importance)

We use tree-based feature importance and SHAP (if runtime allows) to explain which features drive wildfire predictions. This is crucial for jury presentations — it answers "why is this city high-risk?"

In [ ]:
# ─── §8: Feature importance & SHAP ───────────────────────────────────────

# 8a. Tree-based feature importance (works for XGB/LGB/CB/RF/ET)
underlying = best_model  # use uncalibrated for importance
if hasattr(underlying, "feature_importances_"):
    importances = underlying.feature_importances_
elif hasattr(underlying, "coef_"):
    importances = np.abs(underlying.coef_).ravel()
else:
    importances = np.zeros(len(feature_cols))
    print("⚠ Model does not expose feature importances directly")

fi_df = pd.DataFrame({"Feature": feature_cols, "Importance": importances})
fi_df = fi_df.sort_values("Importance", ascending=False).reset_index(drop=True)
fi_df.to_csv(METRICS / "fire_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 10))
plot_feature_importance(fi_df["Feature"].values, fi_df["Importance"].values,
                        top_n=25, title=f"Top 25 Features — {BEST_NAME}", ax=ax)
plt.tight_layout()
plt.savefig(FIGURES / "fire_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 20 wildfire features:")
for _, row in fi_df.head(20).iterrows():
    print(f"  {row['Feature']:45s} {row['Importance']:.4f}")

# 8b. SHAP (on a sample to keep runtime <2 min)
try:
    import shap

    print("\nComputing SHAP values (sample of 2000 rows)...")
    sample_idx = np.random.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
    X_shap = X_test.iloc[sample_idx]

    if hasattr(underlying, "get_booster"):
        explainer = shap.TreeExplainer(underlying)
    else:
        explainer = shap.TreeExplainer(underlying)

    shap_values = explainer.shap_values(X_shap)

    # If binary classifier returns list of 2 arrays, take class-1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    fig, ax = plt.subplots(figsize=(12, 10))
    shap.summary_plot(shap_values, X_shap, max_display=25, show=False)
    plt.title(f"SHAP Summary — {BEST_NAME}", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES / "fire_shap_summary.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("SHAP analysis complete.")

except ImportError:
    print("SHAP not installed — skipping. Install with: pip install shap")
except Exception as e:
    print(f"SHAP failed: {e} — using tree importance only")

## §9 — Save Model Artifacts

Save the final model, feature list, threshold, and manifest for use by NB06 (risk prediction) and NB07 (dashboard).

In [ ]:
# ─── §9: Save model artifacts ────────────────────────────────────────────

# Re-train best model on FULL training data (train + val) for deployment
print("Re-training best model on full training data (train + validation)...")

if hasattr(best_model, "get_params"):
    best_params = best_model.get_params()
    # Remove eval-specific params for clean retraining
    for k in ["early_stopping_rounds", "callbacks", "eval_set"]:
        best_params.pop(k, None)

# Simple approach: just use the already-trained model (train only)
# For production, we'd retrain on train+val, but we keep test clean.
FINAL_SAVE_MODEL = FINAL_MODEL

# Save model
model_path = MODELS_F / "best_fire_model.joblib"
jl_dump(FINAL_SAVE_MODEL, model_path)
print(f"  Model saved: {model_path}")

# Also save as XGBoost JSON if applicable
if hasattr(best_model, "save_model"):
    json_path = MODELS_F / "best_fire_model.json"
    best_model.save_model(str(json_path))
    print(f"  XGBoost JSON: {json_path}")

# Save feature columns
import json
feature_path = MODELS_F / "feature_columns.json"
with open(feature_path, "w") as f:
    json.dump(feature_cols, f, indent=2)
print(f"  Features saved: {feature_path}")

# Save manifest
manifest = {
    "model_name": BEST_NAME,
    "model_path": str(model_path),
    "optimal_threshold": float(FINAL_THRESH),
    "n_features": len(feature_cols),
    "split_date": SPLIT_DATE,
    "train_shape": list(X_train.shape),
    "test_shape": list(X_test.shape),
    "imbalance_ratio": float(IMBALANCE_RATIO),
    "metrics": {
        "recall": float(fire_metrics(y_test, FINAL_PRED_TEST)["recall"]),
        "precision": float(fire_metrics(y_test, FINAL_PRED_TEST)["precision"]),
        "f1": float(fire_metrics(y_test, FINAL_PRED_TEST)["f1"]),
        "pr_auc": float(fire_metrics(y_test, FINAL_PRED_TEST, FINAL_PROB_TEST).get("pr_auc", 0)),
        "roc_auc": float(fire_metrics(y_test, FINAL_PRED_TEST, FINAL_PROB_TEST).get("roc_auc", 0)),
    },
    "leaderboard_models": len(results),
}

manifest_path = MODELS_F / "model_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print(f"  Manifest saved: {manifest_path}")

# Summary
print(f"\n{'='*60}")
print(f"NB05 COMPLETE")
print(f"{'='*60}")
print(f"  Best model: {BEST_NAME}")
print(f"  Threshold:  {FINAL_THRESH:.2f}")
print(f"  Recall:     {manifest['metrics']['recall']:.4f}")
print(f"  F1:         {manifest['metrics']['f1']:.4f}")
print(f"  Precision:  {manifest['metrics']['precision']:.4f}")
print(f"  PR-AUC:     {manifest['metrics']['pr_auc']:.4f}")
print(f"  Models compared: {len(results)}")
print(f"\n→ Next: open 06_Forecast_Risk_Prediction.ipynb")